# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!pip -q uninstall -y torchao   # peft 0.21 rejects Colab's torchao 0.10 when adding LoRA to fp16 nn.Linear (ViT)

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Section 1 checks (GUIDE step 1): counts, session leakage, boxes per class, visualize 6 train images
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

DATA = Path('data')
def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

splits = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in ('train', 'val', 'test')}
meta = json.load(open(DATA / 'splits' / 'meta.json', encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])

# 1) images / boxes per split
for s, rows in splits.items():
    boxes = Counter(o['label'] for r in rows for o in r['objects'])
    sizes = Counter((r['width'], r['height']) for r in rows)
    n_empty = sum(1 for r in rows if not r['objects'])
    n_sess = len({r['session'] for r in rows})
    print(f"{s:5s} images={len(rows):4d} sessions={n_sess:3d} empty_images={n_empty:3d} "
          f"boxes={dict(boxes)} total_boxes={sum(boxes.values())} sizes={dict(sizes)}")
    assert len(rows) == meta['counts'][s]['images']

# 2) no session (and no image) shared between splits
sess = {s: {r['session'] for r in rows} for s, rows in splits.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    inter = sess[a] & sess[b]
    print(f'sessions {a} & {b}: {len(inter)}')
    assert not inter, inter
ids = [r['image_id'] for rows in splits.values() for r in rows]
assert len(ids) == len(set(ids)), 'duplicate image_id across splits'
print('OK: no session / image_id shared between splits')

# 3) box size: median box area as % of frame (small boxes are hard)
areas = sorted(o['yolo'][2] * o['yolo'][3] for r in splits['train'] for o in r['objects'])
print(f"train median box area = {100 * areas[len(areas) // 2]:.2f}% of frame")

# 4) draw 6 train images with GT boxes (YOLO xc,yc,w,h normalized -> pixel corners)
COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
random.seed(0)
sample = random.sample([r for r in splits['train'] if r['objects']], 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, sample):
    img = Image.open(DATA / r['image_path']).convert('RGB')
    W, H = img.size
    d = ImageDraw.Draw(img)
    for o in r['objects']:
        xc, yc, w, h = o['yolo']
        box = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 3, box[1] + 3), o['label'], fill=COLORS[o['label']])
    ax.imshow(img)
    ax.set_title(f"{r['image_id']} | {r['session']} | {len(r['objects'])} boxes", fontsize=9)
    ax.axis('off')
plt.tight_layout()
Path('report/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('report/figures/step1_train_samples.png', dpi=80)
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Section 2 (GUIDE step 2): YOLO labels -> JSON answer on the 0-1000 scale + prompt
import json
from scripts.score import parse_output, iou_xyxy   # the referee's own parser / IoU (read-only use)

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo                                   # normalized [0,1], center + size
    x1, y1, x2, y2 = (min(1000, max(0, round(v * 1000)))
                      for v in (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2))
    # keep tiny boxes valid: score.parse_output rejects the WHOLE answer if any x2<=x1 or y2<=y1
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < 1000 else (999, 1000)
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < 1000 else (999, 1000)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))   # top->bottom, left->right: stable order
    return json.dumps(boxes, separators=(',', ':'))                 # "[]" if no object

PROMPT = (
    "Detect all road damage in this dashcam image. Classes: pothole, crack, manhole.\n"
    "Return ONLY a JSON list, with no extra text and no markdown, in the form "
    '[{"label": "pothole|crack|manhole", "bbox_2d": [x1, y1, x2, y2]}].\n'
    "bbox_2d coordinates are integers on a 0-1000 scale relative to the image width and height "
    "(x1,y1 = top-left corner, x2,y2 = bottom-right corner). If there is no damage, return []."
)

# Check 1: every answer in every split parses with the referee's parser, keeps all boxes,
# and the integer 0-1000 box stays close to the exact GT box the referee uses
worst_iou, worst_px, n_boxes, lens = 1.0, 0.0, 0, []
for s, rows in splits.items():
    for r in rows:
        ans = objects_to_answer(r['objects'])
        lens.append(len(ans))
        parsed = parse_output(ans)
        assert parsed is not None and len(parsed) == len(r['objects']), (s, r['image_id'], ans)
        for o in r['objects']:
            xc, yc, w, h = o['yolo']
            exact = ((xc - w / 2) * 1000, (yc - h / 2) * 1000, (xc + w / 2) * 1000, (yc + h / 2) * 1000)
            got = yolo_to_box(o['label'], o['yolo'])['bbox_2d']
            worst_iou = min(worst_iou, iou_xyxy(exact, tuple(got)))
            scale = (r['width'], r['height'], r['width'], r['height'])
            worst_px = max(worst_px, max(abs(g - e) * k / 1000 for g, e, k in zip(got, exact, scale)))
            n_boxes += 1
print(f"{n_boxes} boxes in {len(lens)} answers all parse OK with score.parse_output")
print(f"worst IoU(int 0-1000 box, exact GT) = {worst_iou:.3f}; worst corner error = {worst_px:.2f} px")
print(f"answer length: max={max(lens)} chars, mean={sum(lens) / len(lens):.0f} chars")

# Check 2: round trip on the busiest train image: JSON 0-1000 -> pixel must sit on the original YOLO box
r = max(splits['train'], key=lambda r: len(r['objects']))
ans = objects_to_answer(r['objects'])
print('\nPROMPT:\n' + PROMPT)
print(f"\nanswer for {r['image_id']} ({len(r['objects'])} boxes):\n{ans}")
img = Image.open(DATA / r['image_path']).convert('RGB')
W, H = img.size
d = ImageDraw.Draw(img)
for o in r['objects']:                          # original YOLO -> pixel (thick green)
    xc, yc, w, h = o['yolo']
    d.rectangle([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], outline='lime', width=5)
for b in json.loads(ans):                       # JSON 0-1000 -> pixel (thin red), should lie on the green box
    x1, y1, x2, y2 = b['bbox_2d']
    d.rectangle([x1 * W / 1000, y1 * H / 1000, x2 * W / 1000, y2 * H / 1000], outline='red', width=1)
    d.text((x1 * W / 1000 + 3, y1 * H / 1000 + 3), b['label'], fill='red')
plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.axis('off')
plt.title(f"Round trip {r['image_id']}: green = original YOLO box, red = JSON 0-1000 -> pixel")
plt.savefig('report/figures/step2_roundtrip.png', dpi=80, bbox_inches='tight')
plt.show()

In [ ]:
# Diagnose GT boxes whose integer 0-1000 version differs from the exact box the referee uses
# Found: 1 zero-height crack box (train, vlcsnap-00058) and 45 boxes extending outside the frame
bad = []
for s, rows in splits.items():
    for r in rows:
        for o in r['objects']:
            xc, yc, w, h = o['yolo']
            exact = ((xc - w / 2) * 1000, (yc - h / 2) * 1000, (xc + w / 2) * 1000, (yc + h / 2) * 1000)
            got = tuple(yolo_to_box(o['label'], o['yolo'])['bbox_2d'])
            v = iou_xyxy(exact, got)
            if v < 0.9:
                bad.append((round(v, 3), s, r['image_id'], o['label'], [round(x, 4) for x in o['yolo']],
                            [round(x, 1) for x in exact], list(got)))
print(len(bad), 'boxes with IoU(int, exact) < 0.9')
for b in sorted(bad)[:20]:
    print(b)
out_of_frame = sum(1 for rows in splits.values() for r in rows for o in r['objects']
                   if min(o['yolo'][0] - o['yolo'][2] / 2, o['yolo'][1] - o['yolo'][3] / 2) < -1e-6
                   or max(o['yolo'][0] + o['yolo'][2] / 2, o['yolo'][1] + o['yolo'][3] / 2) > 1 + 1e-6)
zero_size = sum(1 for rows in splits.values() for r in rows for o in r['objects'] if o['yolo'][2] <= 0 or o['yolo'][3] <= 0)
print('boxes extending outside the frame:', out_of_frame, '| zero-size boxes:', zero_size)

## 3. E0: zero-shot

In [ ]:
# Section 3 (GUIDE step 3): E0 zero-shot — load Qwen3-VL-4B in 4-bit (ViT + lm_head kept fp16)
import gc, time
import bitsandbytes as bnb
import torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
import transformers
transformers.logging.set_verbosity_error()          # silence the per-batch 'processor_kwargs' warning flood

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
Path('work').mkdir(exist_ok=True)

def load_model():
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
                               llm_int8_skip_modules=['model.visual', 'lm_head'])  # re.match from the start: 'visual' alone does NOT match
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={'': 0})
    n4 = sum(isinstance(m, bnb.nn.Linear4bit) for m in model.model.visual.modules())
    assert n4 == 0, f'{n4} ViT/projector Linear layers were quantized to 4-bit'
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = 'left'                 # left pad for batched generation
    return model, processor

def build_messages(img, prompt=PROMPT):
    return [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': prompt}]}]

@torch.no_grad()
def predict(model, processor, rows, out_path, batch_size=4, max_new_tokens=256, prompt=PROMPT):
    """Greedy-decode every row, append {"image_id","raw"} per line. Resume-safe: skips ids already in out_path."""
    out_path = Path(out_path)
    done = {json.loads(l)['image_id'] for l in open(out_path, encoding='utf-8')} if out_path.exists() else set()
    todo = [r for r in rows if r['image_id'] not in done]
    model.eval()
    with open(out_path, 'a', encoding='utf-8') as f:
        for i in tqdm(range(0, len(todo), batch_size), desc=out_path.name):
            batch = todo[i:i + batch_size]
            convs = [build_messages(Image.open(DATA / r['image_path']).convert('RGB'), prompt) for r in batch]
            inputs = processor.apply_chat_template(convs, tokenize=True, add_generation_prompt=True,
                                                   return_dict=True, return_tensors='pt',
                                                   padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            raws = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            for r, raw in zip(batch, raws):
                f.write(json.dumps({'image_id': r['image_id'], 'raw': raw}, ensure_ascii=False) + '\n')
            f.flush()
    preds = [json.loads(l) for l in open(out_path, encoding='utf-8')]
    ids = [p['image_id'] for p in preds]
    assert len(ids) == len(set(ids)), 'duplicate image_id in predictions'
    assert {r['image_id'] for r in rows} <= set(ids), 'missing predictions'
    return preds

t0 = time.time()
model, processor = load_model()
print(f'model loaded in {time.time() - t0:.0f}s | GPU allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('visual dtype:', next(model.model.visual.parameters()).dtype, '| lm_head dtype:', model.lm_head.weight.dtype)

In [ ]:
# E0 inference: quick sanity on 4 val images, then the full test split (311 images) -> work/e0.jsonl
t0 = time.time()
sanity = predict(model, processor, splits['val'][:4], 'work/e0_sanity_val.jsonl')
print(f'4 val images in {time.time() - t0:.1f}s')
for p in sanity:
    print(p['image_id'], '->', repr(p['raw'][:300]))

if not Path('work/e0.jsonl').exists():                 # timing/VRAM only meaningful for a fresh (non-resumed) run
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    e0_preds = predict(model, processor, splits['test'], 'work/e0.jsonl')
    e0_run = {'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 1e9, 2), 'infer_minutes': round((time.time() - t0) / 60, 1)}
    json.dump(e0_run, open('work/e0_run.json', 'w'))
e0_preds = predict(model, processor, splits['test'], 'work/e0.jsonl')   # no-op if complete; asserts no missing ids
print(f"E0: {len(e0_preds)} predictions |", json.load(open('work/e0_run.json')))

In [ ]:
# Score E0 with the referee, then inspect raw outputs (for REPORT question 2)
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
# same predictions read as PIXEL coordinates: does the base model answer in pixels instead of 0-1000? (question 3)
!python scripts/score.py --pred work/e0.jsonl --name E0-as-pixel --coord pixel --out work/e0_as_pixel.json

e0 = [json.loads(l) for l in open('work/e0.jsonl', encoding='utf-8')]
parsed = [parse_output(p['raw']) for p in e0]
boxes = [b for ps in parsed if ps for b in ps]
print(f"\nparse_valid_rate={sum(ps is not None for ps in parsed) / len(e0):.3f} | "
      f"fenced with ```: {sum('```' in p['raw'] for p in e0)} | "
      f"empty []: {sum(ps == [] for ps in parsed)} | predicted boxes: {len(boxes)} (GT test boxes: 583)")
print('predicted labels:', dict(Counter(b['label'] for b in boxes)))
if boxes:
    mx = [max(b['bbox']) for b in boxes]
    print(f"max coordinate per box: median={sorted(mx)[len(mx) // 2]:.0f}, max={max(mx):.0f}; "
          f"share of boxes entirely within 640x360: {sum(b['bbox'][2] <= 640 and b['bbox'][3] <= 360 for b in boxes) / len(boxes):.2f}")
print('\n--- 5 raw outputs ---')
for p in e0[:5]:
    print(p['image_id'], '->', repr(p['raw'][:400]))
bad = [p for p, ps in zip(e0, parsed) if ps is None]
print(f'\n--- {len(bad)} unparseable outputs, first 3 ---')
for p in bad[:3]:
    print(p['image_id'], '->', repr(p['raw'][:400]))

In [ ]:
# E0 resource log (written by the inference cell for a fresh run)
print(json.load(open('work/e0_run.json')))
!ls -la report/evidence work

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Section 4 (GUIDE step 4): chat-template training example, loss mask, collate
import torch.nn.functional as F

IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')

def encode_example(image, prompt, answer):
    user = {'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}
    assistant = {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]}
    prompt_enc = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors='pt')
    full_enc = processor.apply_chat_template([user, assistant], tokenize=True, add_generation_prompt=False,
                                             return_dict=True, return_tensors='pt')
    n = prompt_enc['input_ids'].shape[1]
    assert torch.equal(full_enc['input_ids'][0, :n], prompt_enc['input_ids'][0]), 'prompt is not a prefix of the full chat'
    labels = full_enc['input_ids'].clone()
    labels[:, :n] = -100                                   # loss only on the answer (+ <|im_end|>)
    full_enc['labels'] = labels
    return full_enc

SEQ_KEYS = ('input_ids', 'attention_mask', 'mm_token_type_ids', 'labels')
def collate(batch):
    L = max(b['input_ids'].shape[1] for b in batch)
    pad_value = {'input_ids': processor.tokenizer.pad_token_id, 'attention_mask': 0, 'mm_token_type_ids': 0, 'labels': -100}
    out = {k: torch.cat([F.pad(b[k], (0, L - b[k].shape[1]), value=pad_value[k]) for b in batch])
           for k in SEQ_KEYS if k in batch[0]}
    out['pixel_values'] = torch.cat([b['pixel_values'] for b in batch])        # concatenated, NOT padded
    out['image_grid_thw'] = torch.cat([b['image_grid_thw'] for b in batch])
    return out

# Check 1: one train example -> 220 image tokens, labels decode to exactly the JSON answer + <|im_end|>
r = max(splits['train'][:50], key=lambda r: len(r['objects']))
answer = objects_to_answer(r['objects'])
ex = encode_example(Image.open(DATA / r['image_path']).convert('RGB'), PROMPT, answer)
seq = ex['input_ids'].shape[1]
n_img = int((ex['input_ids'] == IMAGE_TOKEN_ID).sum())
ans_ids = ex['labels'][0][ex['labels'][0] != -100]
decoded = processor.tokenizer.decode(ans_ids)
print('keys:', list(ex.keys()), '| image_grid_thw:', ex['image_grid_thw'].tolist())
print(f'sequence={seq} tokens | image tokens={n_img} | loss tokens={len(ans_ids)} ({100 * len(ans_ids) / seq:.1f}% of sequence)')
print('decoded labels:', repr(decoded))
assert n_img == 220, n_img
assert decoded.startswith(answer + '<|im_end|>'), decoded
unknown = set(ex.keys()) - set(SEQ_KEYS) - {'pixel_values', 'image_grid_thw'}
assert not unknown, f'collate ignores keys {unknown}'

# Check 2: collate 2 examples of different length, then one no-grad forward pass gives a finite loss
r2 = next(r2 for r2 in splits['train'] if len(r2['objects']) != len(r['objects']))
ex2 = encode_example(Image.open(DATA / r2['image_path']).convert('RGB'), PROMPT, objects_to_answer(r2['objects']))
batch = collate([ex, ex2])
print('batch shapes:', {k: tuple(v.shape) for k, v in batch.items()})
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    loss = model(**{k: v.to(model.device) for k, v in batch.items()}).loss
print(f'zero-shot loss on 2 train examples = {loss.item():.3f}')
assert torch.isfinite(loss)

# Check 3: answer length in tokens over train (is max_new_tokens=256 enough at inference?)
tok_lens = sorted(len(processor.tokenizer(objects_to_answer(r['objects']))['input_ids']) + 1 for r in splits['train'])
print(f"answer tokens (+<|im_end|>): median={tok_lens[len(tok_lens) // 2]}, p99={tok_lens[int(0.99 * len(tok_lens))]}, "
      f"max={tok_lens[-1]}, over 256: {sum(t > 256 for t in tok_lens)}")

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Section 5 (GUIDE step 5): LoRA / freezing setups E1, E2, E3
from peft import LoraConfig, get_peft_model
from peft.tuners.lora import LoraLayer

LLM_RE = r'.*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)'
VIT_RE = r'.*visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.linear_fc(1|2))'
PROJ_KEYS = ('visual.merger', 'visual.deepstack_merger_list')

def apply_setup(model, setup, r=16, alpha=32, dropout=0.05):
    """E1: LoRA on LLM. E2: + full projector (fp32). E3: E2 + LoRA on ViT blocks."""
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()                    # needed with gradient checkpointing
    target = LLM_RE if setup in ('E1', 'E2') else f'({LLM_RE})|({VIT_RE})'
    model = get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=target))
    if setup in ('E2', 'E3'):
        for name, p in model.named_parameters():
            if any(k in name for k in PROJ_KEYS):
                p.data = p.data.float()                   # fp32 master weights for full training
                p.requires_grad = True
    lora_modules = [n for n, m in model.named_modules() if isinstance(m, LoraLayer)]
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{setup}: LoRA modules={len(lora_modules)} | trainable params={trainable:,} ({trainable / 1e6:.1f}M)')
    return model, lora_modules, trainable

def n_params(p):                                          # bnb 4-bit weights are packed 2 per byte
    return p.numel() * 2 if p.__class__.__name__ == 'Params4bit' else p.numel()

# free the E0 model(s), then build each setup on a fresh 4-bit model
del model
gc.collect(); torch.cuda.empty_cache()

setup_stats = {}
for setup in ('E1', 'E2', 'E3'):
    base, processor = load_model()
    if setup == 'E1':                                     # parameter count per block (REPORT question 1)
        blocks = Counter()
        for name, p in base.named_parameters():
            key = ('projector: visual.merger' if 'visual.merger' in name else
                   'projector: visual.deepstack_merger_list' if 'deepstack_merger_list' in name else
                   'ViT (visual, without projector)' if '.visual.' in name or name.startswith('visual') else
                   'lm_head' if 'lm_head' in name else 'LLM (language_model + embeddings)')
            blocks[key] += n_params(p)
        for k, v in blocks.items():
            print(f'  {k:42s} {v / 1e6:9.1f}M')
        print(f"  {'total':42s} {sum(blocks.values()) / 1e6:9.1f}M | tie_word_embeddings={base.config.get_text_config().tie_word_embeddings}")
    pm, mods, trainable = apply_setup(base, setup)
    setup_stats[setup] = {'lora_modules': len(mods), 'trainable_params': trainable}
    del pm, base
    gc.collect(); torch.cuda.empty_cache()

print(setup_stats)
assert [setup_stats[s]['lora_modules'] for s in ('E1', 'E2', 'E3')] == [252, 252, 348]
print(f"E2 - E1 = {(setup_stats['E2']['trainable_params'] - setup_stats['E1']['trainable_params']) / 1e6:.1f}M (projector)")
json.dump(setup_stats, open('work/setup_stats.json', 'w'))

In [ ]:
# Diagnose: is the ViT / projector really kept out of 4-bit quantization?
# First run (with llm_int8_skip_modules=['visual', ...]) showed 104 Linear4bit layers in model.visual and E2-E1 = 54.6M:
# transformers 5.19 matches skip patterns with re.match from the START of the module name, so 'visual' never matches
# 'model.visual.blocks...'. Fixed in load_model() with 'model.visual'.
import inspect, bitsandbytes as bnb, transformers
import transformers.quantizers.quantizers_utils as qu
base, processor = load_model()
def kinds(prefix):
    c = Counter()
    for n, m in base.named_modules():
        if n.startswith(prefix) and isinstance(m, (torch.nn.Linear, bnb.nn.Linear4bit)):
            c[type(m).__name__] += 1
    return dict(c)
print('transformers', transformers.__version__)
print('model.visual           :', kinds('model.visual.'))
print('  merger               :', kinds('model.visual.merger'))
print('  deepstack_merger_list:', kinds('model.visual.deepstack_merger_list'))
print('model.language_model   :', kinds('model.language_model.'))
print('lm_head                :', type(base.lm_head).__name__)
print(inspect.getsource(qu.should_convert_module))
for pats in (['visual'], ['model.visual'], ['visual.'], [r'.*visual.*']):
    print(pats, '->', 'blocks.0.attn.qkv quantized' if qu.should_convert_module('model.visual.blocks.0.attn.qkv', pats) else 'kept fp16',
          '|', 'merger quantized' if qu.should_convert_module('model.visual.merger.linear_fc1', pats) else 'kept fp16',
          '|', 'LLM q_proj quantized' if qu.should_convert_module('model.language_model.layers.0.self_attn.q_proj', pats) else 'LLM kept fp16 (!)')
del base
gc.collect(); torch.cuda.empty_cache()

## 6. Vòng huấn luyện

In [ ]:
# Section 7a (GUIDE step 6): ONE train() for E1/E2/E3 (and E4), hyper-parameters read from configs/eN.yaml
# fp16 autocast + GradScaler, gradient checkpointing (non-reentrant), AdamW, 5% warmup + cosine to 0, clip 1.0
import math, os, yaml
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from torch.utils.data import DataLoader, Dataset
from transformers import get_cosine_schedule_with_warmup

def answer_norm1000(r):
    return objects_to_answer(r['objects'])

class SFTDataset(Dataset):
    """train rows -> encode_example(image, prompt, answer): loss only on the answer + <|im_end|> (section 6a)"""
    def __init__(self, rows, prompt=PROMPT, to_answer=answer_norm1000):
        self.rows, self.prompt, self.to_answer = rows, prompt, to_answer
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return encode_example(Image.open(DATA / r['image_path']).convert('RGB'), self.prompt, self.to_answer(r))

def save_trainable(model, out_dir):
    """peft saves ONLY the LoRA adapter -> the fully trained projector (E2/E3) is saved separately."""
    model.save_pretrained(f'{out_dir}_adapter')
    extra = {n: p.detach().cpu() for n, p in model.named_parameters()
             if p.requires_grad and any(k in n for k in PROJ_KEYS)}
    if extra:
        torch.save(extra, f'{out_dir}_extra_trainable.pt')
    return len(extra)

def train(setup, out_dir, prompt=PROMPT, to_answer=answer_norm1000, log_every=25, **overrides):
    cfg = yaml.safe_load(open(f'configs/{setup.lower()}.yaml'))
    hp = {'r': cfg['lora']['r'], 'alpha': cfg['lora']['alpha'], 'dropout': cfg['lora']['dropout'], 'lr': cfg['lr'],
          'batch_size': cfg['batch_size'], 'grad_accum': cfg['grad_accum'], 'max_steps': cfg['max_steps'], 'seed': cfg['seed']}
    hp.update(overrides)
    print(f'== train {setup} -> {out_dir} | {hp}')
    random.seed(hp['seed']); torch.manual_seed(hp['seed'])
    base, _ = load_model()
    model, lora_modules, trainable = apply_setup(base, setup, r=hp['r'], alpha=hp['alpha'], dropout=hp['dropout'])
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.config.use_cache = False
    model.config.get_text_config().use_cache = False
    params = [p for p in model.parameters() if p.requires_grad]
    assert sum(p.numel() for p in params) == trainable
    assert all(p.dtype == torch.float32 for p in params), 'trainable params must be fp32 (GradScaler / AdamW master weights)'

    loader = DataLoader(SFTDataset(splits['train'], prompt, to_answer), batch_size=hp['batch_size'], shuffle=True,
                        drop_last=True, collate_fn=collate, num_workers=2, multiprocessing_context='fork',
                        persistent_workers=True, generator=torch.Generator().manual_seed(hp['seed']))
    def batches():                                        # endless stream of batches, reshuffled every epoch
        while True:
            yield from loader
    it = batches()
    opt = torch.optim.AdamW(params, lr=hp['lr'], weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, math.ceil(0.05 * hp['max_steps']), hp['max_steps'])
    scaler = torch.amp.GradScaler('cuda')
    model.train()
    gc.collect(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()                  # peak VRAM of training only (model weights included)
    log, n_extra, t0 = [], 0, time.time()
    for step in range(1, hp['max_steps'] + 1):
        step_loss = 0.0
        for _ in range(hp['grad_accum']):
            batch = {k: v.to(model.device) for k, v in next(it).items()}
            with torch.autocast('cuda', dtype=torch.float16):
                loss = model(**batch).loss / hp['grad_accum']
            if not torch.isfinite(loss):
                raise FloatingPointError(f'{setup} step {step}: loss = {loss.item()} (fp16 overflow?) -> stop')
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(opt)
        grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        sched.step()
        log.append({'step': step, 'loss': round(step_loss, 4), 'grad_norm': round(grad_norm, 3), 'lr': sched.get_last_lr()[0]})
        if step == 1 or step % log_every == 0:
            el = time.time() - t0
            print(f"step {step:3d}/{hp['max_steps']} loss={step_loss:.4f} grad_norm={grad_norm:.2f} lr={log[-1]['lr']:.2e} "
                  f"scale={scaler.get_scale():.0f} | {el / 60:.1f} min, {el / step:.2f} s/step, "
                  f"peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB", flush=True)
        if step % 100 == 0 or step == hp['max_steps']:
            n_extra = save_trainable(model, out_dir)      # checkpoint every 100 steps: adapter + projector
    run = {'setup': setup, 'trainable_params': trainable,
           'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 1e9, 2),
           'train_minutes': round((time.time() - t0) / 60, 1), 'lora_modules': len(lora_modules),
           'projector_tensors_saved': n_extra, 'final_loss_last10': round(sum(l['loss'] for l in log[-10:]) / len(log[-10:]), 4),
           'gpu': torch.cuda.get_device_name(0), **hp}
    json.dump(run, open(f'{out_dir}_run.json', 'w'), indent=1)
    json.dump(log, open(f'{out_dir}_loss.json', 'w'))
    print('run:', run)
    return run

## 7. Suy luận và chấm E1–E3

In [ ]:
# Section 7b (GUIDE step 7): reload base 4-bit + LoRA adapter (+ projector for E2/E3), debug on 40 val images,
# then predict the full test split ONCE and score it with --meta from work/eN_run.json
import shutil, subprocess
from peft import PeftModel

def load_trained(out_dir):
    base, proc = load_model()
    model = PeftModel.from_pretrained(base, f'{out_dir}_adapter')
    extra_path = Path(f'{out_dir}_extra_trainable.pt')
    if extra_path.exists():                               # E2/E3: the projector is NOT inside the peft adapter
        extra = torch.load(extra_path, map_location='cpu')
        params = dict(model.named_parameters())
        k0 = next(iter(extra))
        before = params[k0].detach().float().cpu().clone()
        res = model.load_state_dict(extra, strict=False)
        assert not res.unexpected_keys, f'projector keys do not match the model: {res.unexpected_keys[:3]}'
        after = params[k0].detach().float().cpu()
        assert torch.equal(after, extra[k0].half().float()) and not torch.equal(after, before), 'projector not loaded'
        print(f'loaded {len(extra)} projector tensors from {extra_path.name}: unexpected_keys=0, '
              f'mean |trained - base| of {k0} = {(after - before).abs().mean():.2e}')
    model.eval()
    return model, proc

def run_score(pred, name, out, split='test', coord='norm1000', meta=None, splits_dir='data/splits'):
    cmd = ['python', 'scripts/score.py', '--pred', pred, '--name', name, '--out', out,
           '--split', split, '--coord', coord, '--splits-dir', splits_dir]
    for k, v in (meta or {}).items():
        cmd += ['--meta', f'{k}={v}']
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout.strip(), res.stderr.strip()[-300:])
    res.check_returncode()
    return json.load(open(out, encoding='utf-8'))

# debug split = the first 40 val images, in its own splits dir so the referee scores them without 'missing' noise
VAL40 = Path('work/val40')
VAL40.mkdir(parents=True, exist_ok=True)
with open(VAL40 / 'val.jsonl', 'w', encoding='utf-8') as f:
    for r in splits['val'][:40]:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
shutil.copy(DATA / 'splits' / 'meta.json', VAL40 / 'meta.json')

META_KEYS = ('trainable_params', 'peak_vram_gb', 'train_minutes')

def run_setup(setup, out_dir, name, evidence, prompt=PROMPT, to_answer=answer_norm1000, coord='norm1000', **overrides):
    """train (skipped if out_dir_run.json exists) -> reload -> 40 val images (debug) -> full test once -> score + meta"""
    if not Path(f'{out_dir}_run.json').exists():
        train(setup, out_dir, prompt=prompt, to_answer=to_answer, **overrides)
        gc.collect(); torch.cuda.empty_cache()
    run = json.load(open(f'{out_dir}_run.json'))
    model, proc = load_trained(out_dir)
    predict(model, proc, splits['val'][:40], f'{out_dir}_val.jsonl', prompt=prompt)
    val = run_score(f'{out_dir}_val.jsonl', f'{name}-val40', f'{out_dir}_val.json', split='val', coord=coord,
                    splits_dir=str(VAL40))
    assert val['parse_valid_rate'] >= 0.8, 'outputs mostly unparseable on val: fix before scoring test'
    fresh, t0 = not Path(f'{out_dir}.jsonl').exists(), time.time()
    predict(model, proc, splits['test'], f'{out_dir}.jsonl', prompt=prompt)
    if fresh:
        run['infer_minutes'] = round((time.time() - t0) / 60, 1)
        json.dump(run, open(f'{out_dir}_run.json', 'w'), indent=1)
    del model, proc
    gc.collect(); torch.cuda.empty_cache()
    ev = run_score(f'{out_dir}.jsonl', name, evidence, coord=coord, meta={k: run[k] for k in META_KEYS})
    print(f"{name}: valid={ev['parse_valid_rate']:.3f} F1={ev['micro']['f1']:.3f} "
          f"CI95={ev['f1_ci95'][0]:.3f}-{ev['f1_ci95'][1]:.3f} | meta={ev['meta']}")
    return run, ev

In [ ]:
# Smoke test before the real runs: 2 optimizer steps of E3 (LoRA LLM + LoRA ViT + fp32 projector),
# save, reload (projector included), generate on 4 val images. Catches dtype / save-load bugs in ~3 minutes.
smoke = train('E3', 'work/smoke_e3', max_steps=2, log_every=1)
gc.collect(); torch.cuda.empty_cache()
m_, p_ = load_trained('work/smoke_e3')
for x in predict(m_, p_, splits['val'][:4], 'work/smoke_e3_val.jsonl'):
    print(x['image_id'], '->', repr(x['raw'][:150]))
del m_, p_
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Section 7c: E1 = LoRA r=16 on the 252 LLM linear layers; ViT + projector frozen
run_e1, ev_e1 = run_setup('E1', 'work/e1', 'E1', 'report/evidence/e1.json')
ev_e0 = json.load(open('report/evidence/e0.json'))
gain = ev_e1['micro']['f1'] - ev_e0['micro']['f1']
print(f"check 7.3: parse_valid_rate(E1) = {ev_e1['parse_valid_rate']:.3f} (need >= 0.95) | "
      f"F1(E1) - F1(E0) = {gain:.3f} (need >= 0.10)")
assert ev_e1['parse_valid_rate'] >= 0.95 and gain >= 0.10

In [ ]:
# Section 7d: E2 = E1 + full training of the projector (visual.merger + visual.deepstack_merger_list, fp32)
run_e2, ev_e2 = run_setup('E2', 'work/e2', 'E2', 'report/evidence/e2.json')

In [ ]:
# Section 7e: E3 = E2 + LoRA r=16 on the 96 ViT attention/MLP linear layers (348 LoRA modules in total)
run_e3, ev_e3 = run_setup('E3', 'work/e3', 'E3', 'report/evidence/e3.json')

In [ ]:
# Section 7 summary: E0-E3 table straight from report/evidence/*.json, checks 7.3, loss curves
EV = {s: json.load(open(f'report/evidence/{s.lower()}.json', encoding='utf-8')) for s in ('E0', 'E1', 'E2', 'E3')}
print('| setup | trainable | VRAM GB | train min | valid | F1 | CI95 | FP/img | pothole | crack | manhole |')
for s, e in EV.items():
    m = e['meta']
    print(f"| {s} | {m.get('trainable_params', 0):,} | {m.get('peak_vram_gb', '-')} | {m.get('train_minutes', '-')} | "
          f"{e['parse_valid_rate']:.3f} | {e['micro']['f1']:.3f} | {e['f1_ci95'][0]:.3f}-{e['f1_ci95'][1]:.3f} | "
          f"{e['fp_per_image']:.2f} | " + ' | '.join(f"{e['per_class'][c]['f1']:.3f}" for c in ('pothole', 'crack', 'manhole')) + ' |')
for s in ('E1', 'E2', 'E3'):
    assert EV[s]['split'] == 'test' and EV[s]['missing_predictions'] == 0
    assert all(isinstance(EV[s]['meta'].get(k), (int, float)) and EV[s]['meta'][k] > 0 for k in META_KEYS), s
for s in ('E2', 'E3'):                                    # identical to E1 => projector was not reloaded
    assert EV[s]['micro'] != EV['E1']['micro'], f'{s} scores exactly like E1: projector not loaded?'
for a, b in (('E1', 'E2'), ('E2', 'E3'), ('E1', 'E3')):
    (l1, h1), (l2, h2) = EV[a]['f1_ci95'], EV[b]['f1_ci95']
    print(f"{a} vs {b}: dF1 = {EV[b]['micro']['f1'] - EV[a]['micro']['f1']:+.3f}, CI95 overlap = {max(l1, l2) < min(h1, h2)}")
for s in ('E1', 'E2', 'E3'):
    r = json.load(open(f'work/{s.lower()}_run.json'))
    print(s, {k: r.get(k) for k in ('trainable_params', 'peak_vram_gb', 'train_minutes', 'infer_minutes', 'final_loss_last10')})

fig, ax = plt.subplots(figsize=(9, 4))
for s in ('E1', 'E2', 'E3'):
    log = json.load(open(f'work/{s.lower()}_loss.json'))
    y = [l['loss'] for l in log]
    k = 10                                                # moving average over 10 optimizer steps
    ax.plot([l['step'] for l in log][k - 1:], [sum(y[i - k + 1:i + 1]) / k for i in range(k - 1, len(y))], label=s)
ax.set_xlabel('optimizer step (batch 2 x grad_accum 4)'); ax.set_ylabel('train loss (answer tokens, MA10)')
ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend(); ax.set_title('Training loss E1-E3')
plt.tight_layout()
plt.savefig('report/figures/train_loss_e1_e3.png', dpi=90)
plt.show()

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# Section 8a (GUIDE step 8): E4 ablation of ONE variable = coord_scale: norm1000 (as E1-E3) vs pixel.
# Everything else is the best setup of E1-E3 (same blocks trained, LoRA r/alpha, lr, steps, data, seed);
# for 'pixel' the labels, the prompt and the scoring (--coord pixel) all switch to absolute pixels of the 640x360 frame.
EV13 = {s: json.load(open(f'report/evidence/{s.lower()}.json', encoding='utf-8')) for s in ('E1', 'E2', 'E3')}
BEST = max(EV13, key=lambda s: EV13[s]['micro']['f1'])
print('best setup of E1-E3 by test F1:', BEST, {s: round(e['micro']['f1'], 3) for s, e in EV13.items()})
assert {(r['width'], r['height']) for rows in splits.values() for r in rows} == {(640, 360)}

def yolo_to_box_pixel(label, yolo, W, H):
    xc, yc, w, h = yolo
    x1, x2 = (min(W, max(0, round(v * W))) for v in (xc - w / 2, xc + w / 2))
    y1, y2 = (min(H, max(0, round(v * H))) for v in (yc - h / 2, yc + h / 2))
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < W else (W - 1, W)   # same tiny-box guard as yolo_to_box
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < H else (H - 1, H)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def answer_pixel(r):
    boxes = [yolo_to_box_pixel(o['label'], o['yolo'], r['width'], r['height']) for o in r['objects']]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))  # same (y1, x1) order as norm1000
    return json.dumps(boxes, separators=(',', ':'))

OLD = 'bbox_2d coordinates are integers on a 0-1000 scale relative to the image width and height '
NEW = 'bbox_2d coordinates are integer pixel coordinates in the 640x360 image (x from 0 to 640, y from 0 to 360) '
assert PROMPT.count(OLD) == 1
PROMPT_PIXEL = PROMPT.replace(OLD, NEW)
print(PROMPT_PIXEL)

# checks: every pixel answer parses with the referee, and pixel -> 0-1000 (what score.py --coord pixel does)
# lands on the exact GT box as well as the integer 0-1000 box does
ious = {'pixel': [], 'norm1000': []}
for s, rows in splits.items():
    for r in rows:
        parsed = parse_output(answer_pixel(r))
        assert parsed is not None and len(parsed) == len(r['objects']), (s, r['image_id'])
        for o in r['objects']:
            xc, yc, w, h = o['yolo']
            exact = ((xc - w / 2) * 1000, (yc - h / 2) * 1000, (xc + w / 2) * 1000, (yc + h / 2) * 1000)
            x1, y1, x2, y2 = yolo_to_box_pixel(o['label'], o['yolo'], r['width'], r['height'])['bbox_2d']
            ious['pixel'].append(iou_xyxy(exact, (x1 / r['width'] * 1000, y1 / r['height'] * 1000,
                                                  x2 / r['width'] * 1000, y2 / r['height'] * 1000)))
            ious['norm1000'].append(iou_xyxy(exact, tuple(yolo_to_box(o['label'], o['yolo'])['bbox_2d'])))
for k, v in ious.items():
    print(f'{k:8s}: {len(v)} boxes parse OK | worst IoU(label box, exact GT) = {min(v):.3f} | IoU < 0.9: {sum(x < 0.9 for x in v)}')
r = max(splits['train'], key=lambda r: len(r['objects']))
print('norm1000:', answer_norm1000(r)[:150])
print('pixel   :', answer_pixel(r)[:150])

In [ ]:
# Section 8b: E4 runs. pixel = 1 new training of BEST with pixel labels + prompt, scored with --coord pixel.
# norm1000 = the BEST setup itself: same test predictions re-scored under the E4 name (no retraining).
run_e4, ev_e4_pixel = run_setup(BEST, 'work/e4_pixel', 'E4-coord_scale-pixel', 'report/evidence/e4_coord_scale_pixel.json',
                                prompt=PROMPT_PIXEL, to_answer=answer_pixel, coord='pixel')
run_best = json.load(open(f'work/{BEST.lower()}_run.json'))
ev_e4_norm = run_score(f'work/{BEST.lower()}.jsonl', 'E4-coord_scale-norm1000', 'report/evidence/e4_coord_scale_norm1000.json',
                       meta={k: run_best[k] for k in META_KEYS})

# Question 3 (no GPU): read each model's output on the WRONG scale
cross = {
    'E1 output read as pixel': run_score('work/e1.jsonl', 'E1-as-pixel', 'work/e1_as_pixel.json', coord='pixel'),
    f'{BEST} output read as pixel': run_score(f'work/{BEST.lower()}.jsonl', f'{BEST}-as-pixel', f'work/{BEST.lower()}_as_pixel.json', coord='pixel'),
    'E4-pixel output read as norm1000': run_score('work/e4_pixel.jsonl', 'E4-pixel-as-norm1000', 'work/e4_pixel_as_norm1000.json'),
}
print('\n| E4 coord_scale | valid | F1 | CI95 | FP/img | mean IoU | pothole | crack | manhole |')
for k, e in (('norm1000', ev_e4_norm), ('pixel', ev_e4_pixel), *cross.items()):
    print(f"| {k} | {e['parse_valid_rate']:.3f} | {e['micro']['f1']:.3f} | {e['f1_ci95'][0]:.3f}-{e['f1_ci95'][1]:.3f} | "
          f"{e['fp_per_image']:.2f} | {e['mean_iou']:.3f} | " + ' | '.join(f"{e['per_class'][c]['f1']:.3f}" for c in ('pothole', 'crack', 'manhole')) + ' |')
(l1, h1), (l2, h2) = ev_e4_norm['f1_ci95'], ev_e4_pixel['f1_ci95']
print(f"dF1(pixel - norm1000) = {ev_e4_pixel['micro']['f1'] - ev_e4_norm['micro']['f1']:+.3f}, CI95 overlap = {max(l1, l2) < min(h1, h2)}")
print('E4 pixel run:', {k: run_e4.get(k) for k in ('trainable_params', 'peak_vram_gb', 'train_minutes', 'infer_minutes', 'final_loss_last10')})
!ls -la report/evidence

In [ ]:
# YOUR CODE HERE: GUIDE bước 9 (E5, tuỳ chọn)